# StudyLens: train the arrow detector

The weakest stage of the scene graph is the arrow list the VLM writes (arrow F1 about 0.30 on the AI2D dev set). This notebook trains a **Mask R-CNN** (torchvision, starting from COCO weights) to find the drawn **arrows** and **arrowheads**, using AI2D's annotations. A deterministic rule (`studylens/arrows.py`) then attaches each arrow's tail and head to the nearest part.

Measured before training: attaching AI2D's *true* arrow outlines to our scene-graph parts gives arrow F1 0.53 on the dev set (vs 0.30), so a good detector is worth it.

* **GPU:** a **T4** is enough (an L4/A100 is faster). Run the probe cell first: it prints minutes per epoch.
* **Time:** roughly 1–2 h for 12 epochs on a T4. Everything is **resumable**: checkpoints live on Google Drive; after a disconnect, re-run the cells.
* **Licences:** AI2D (AI2 non-commercial licence, no redistribution); torchvision and its COCO weights (BSD). The detector weights are derived from AI2D: keep the Drive folder private and do not publish them.
* Only the AI2D **train** split is used; 150 train diagrams are held out for validation. The AI2D test split is never used here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/studylens'
!mkdir -p $DRIVE/external $DRIVE/checkpoints
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and packages

In [ ]:
REPO_URL = 'https://github.com/krish-patel1003/CECS558-term-project-studylens.git'
import os
if not os.path.exists('/content/studylens'):
    !git clone $REPO_URL /content/studylens
%cd /content/studylens
!git pull -q
!pip install -q -r requirements.txt torchvision

## 2. AI2D and checkpoints on Drive
The AI2D download requires accepting its licence (`--accept-license`). If you already ran the scene-graph notebook, AI2D is already on Drive and is reused.

In [ ]:
!rm -rf external checkpoints && ln -s $DRIVE/external external && ln -s $DRIVE/checkpoints checkpoints
!python scripts/get_ai2d.py --accept-license
!ls external/ai2d | head

## 3. Probe: speed and memory (10 steps, nothing saved)

In [ ]:
!python -u scripts/train_arrow_detector.py --probe --run probe --batch 4

## 4. Train (resumable)
Defaults: 12 epochs, batch 4, SGD lr 0.01 with warm-up and cosine decay, images resized to 640–1024 px, random horizontal flips. A checkpoint is saved after every epoch; if the session disconnects, run this cell again and it resumes. If the probe ran out of memory, use `--batch 2 --lr 0.005`.

In [ ]:
!python -u scripts/train_arrow_detector.py --run arrows --epochs 12 --batch 4

## 5. Validation on the 150 held-out train diagrams
Reports detection precision/recall at IoU 0.5 (arrows, arrowheads) and **arrow F1 on gold parts**: detected arrows attached to the true parts with the same rule StudyLens uses, next to the same rule applied to the true arrow outlines (the ceiling). Writes `checkpoints/arrows/val_metrics.json`. Try `--score 0.3` and `--score 0.7` too; pick the threshold with the best detector F1 on these held-out diagrams.

In [ ]:
!python -u scripts/train_arrow_detector.py --run arrows --eval --score 0.5
!cat checkpoints/arrows/val_metrics.json

## 6. Use the detector
Download `checkpoints/arrows/arrow_detector.pt` from Drive (about 170 MB) into the project's `checkpoints/` folder on the laptop. The scene graph then uses it automatically instead of the VLM's arrow list (set `ARROW_DETECTOR=0` to switch it off). It runs on the CPU in a few seconds per diagram.

Evaluation on the 100 AI2D development diagrams (system `v2d`), on the laptop:

```
python scripts/evaluate_grounding.py --n-parts 60 --n-flow 40 --tag v2d
```